In [1]:
# importing required  libraries
from pyspark.sql import SparkSession, functions as F
from pyspark.storagelevel import StorageLevel

In [2]:
# create the spark session
spark = (SparkSession.builder.appName("NYC_Taxi_Data_Cleaning").getOrCreate())

In [9]:
# load the combined 2024 dataset for tableau dashboarding
df = spark.read.csv("/content/sample_df.csv",header=True,inferSchema=True)
df.show(10, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+-----------+------------------+----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|is_long_trip|pickup_hour|pickup_day_of_week|is_weekend|pickup_month|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+-----------+------------------+--------

In [4]:
print("Original schema:")
df.printSchema()

Original schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- is_long_trip: integer (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_day_of_

In [5]:
count = df.count()
print("rows:", count)
print("columns:", len(df.columns))

rows: 4117091
columns: 24


In [6]:
# cheking duplicate
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()
duplicate_count = total_rows - distinct_rows
print("total rows:", total_rows)
print("distinct rows:", distinct_rows)
print("duplicate rows:", duplicate_count)

total rows: 4117091
distinct rows: 4117091
duplicate rows: 0


In [8]:
# missing value checking
missing_values = df.agg(*[F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
missing_values.show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+-----------+------------------+----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|is_long_trip|pickup_hour|pickup_day_of_week|is_weekend|pickup_month|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+-----------+------------------+--------

In [10]:
# columns to impute using median
median_columns = ["passenger_count","RatecodeID","congestion_surcharge","Airport_fee"]
for col_name in median_columns:
    median_value = df.approxQuantile(col_name, [0.5], 0.01)[0]
    if median_value is not None:
        df = df.fillna({col_name: median_value})
    print(f"{col_name}: median = {median_value}")

passenger_count: median = 1.0
RatecodeID: median = 1.0
congestion_surcharge: median = 2.5
Airport_fee: median = 0.0


In [17]:
# impute missing categorical values using mode
mode_row = (df.filter(F.col("store_and_fwd_flag").isNotNull()).groupBy("store_and_fwd_flag").count().orderBy(F.desc("count")).first())

In [18]:
if mode_row is not None:
    mode_value = mode_row["store_and_fwd_flag"]
    df = df.fillna({"store_and_fwd_flag": mode_value})
    print("Missing values filled with mode:", mode_value)

Missing values filled with mode: N


In [21]:
df.agg(*[F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns]).show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+-----------+------------------+----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|is_long_trip|pickup_hour|pickup_day_of_week|is_weekend|pickup_month|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+-----------+------------------+--------

In [23]:
# convert columns to appropriate data types
df = (df
    .withColumn("tpep_pickup_datetime", F.to_timestamp("tpep_pickup_datetime"))
    .withColumn("tpep_dropoff_datetime", F.to_timestamp("tpep_dropoff_datetime"))
    .withColumn("trip_distance", F.col("trip_distance").cast("double"))
    .withColumn("fare_amount", F.col("fare_amount").cast("double"))
    .withColumn("total_amount", F.col("total_amount").cast("double"))
    .withColumn("payment_type", F.col("payment_type").cast("int")))

In [24]:
# invalid data check
checks = {
    "Missing pickup datetime": F.col("tpep_pickup_datetime").isNull(),
    "Missing drop-off datetime": F.col("tpep_dropoff_datetime").isNull(),
    "Negative trip distance": F.col("trip_distance") < 0,
    "Negative fare amount": F.col("fare_amount") < 0,
    "Negative total amount": F.col("total_amount") < 0,
    "Drop-off before pickup": (F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"))}

In [25]:
for description, condition in checks.items():
    print(description, ":", df.filter(condition).count())

Missing pickup datetime : 0
Missing drop-off datetime : 0
Negative trip distance : 0
Negative fare amount : 73525
Negative total amount : 61189
Drop-off before pickup : 150


In [26]:
# remove only records with invalid trip timing
before_cleaning = df.count()
df = df.filter(F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
after_cleaning = df.count()
print("rows before filtering:", before_cleaning)
print("rows after filtering:", after_cleaning)
print("invalid timestamp rows removed:", before_cleaning - after_cleaning)

rows before filtering: 4117091
rows after filtering: 4116941
invalid timestamp rows removed: 150


In [28]:
# export cleaned dataset as CSV
output_path = "/content/nyc_taxi_cleaned_2024_csv"
df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_path)
print("saved", output_path)

saved /content/nyc_taxi_cleaned_2024_csv
